In [ ]:
import xarray as xr
import pandas as pd
from pathlib import Path
import dask
from dask.diagnostics import ProgressBar
import warnings
from xarray import SerializationWarning
#ignore these warnings
warnings.filterwarnings("ignore", category=SerializationWarning)

/home/m/matthewwright/nobackups/anaconda3/envs/initial-analysis/lib/python3.10/site-packages/xarray/backends/cfgrib_.py:29: UserWarning: Failed to load cfgrib - most likely there is a problem accessing the ecCodes library. Try `import cfgrib` to get the full error message
  warnings.warn(


In [ ]:
BASE_DIR = "XXXX"
OUT_DIR_6H = "XXX"
OUT_DIR_24H = "XXX"
var = "qpe"
var_out = "rainfall_rate"
example_grid_path = None

In [ ]:
if not isinstance(BASE_DIR, Path):
    BASE_DIR = Path(BASE_DIR)

if not isinstance(OUT_DIR_6H, Path):
    OUT_DIR_6H = Path(OUT_DIR_6H)

OUT_DIR_6H.mkdir(parents=True, exist_ok=True)

if not isinstance(OUT_DIR_24H, Path):
    OUT_DIR_24H = Path(OUT_DIR_24H)

OUT_DIR_24H.mkdir(parents=True, exist_ok=True)


if example_grid_path is None:
    x_min, x_max, y_min, y_max = None, None, None, None
else:
    example_grid = xr.load_dataset(example_grid_path)

    if "x" not in example_grid.coords or "y" not in example_grid.coords:
        raise ValueError(
            "example_grid must contain 'x' and 'y' coordinates."
        )

    x_min = float(example_grid["x"].min())
    x_max = float(example_grid["x"].max())
    y_min = float(example_grid["y"].min())
    y_max = float(example_grid["y"].max())

    example_grid.close()

    print(
        f"Crop bounds: "
        f"x={x_min:.3f} to {x_max:.3f}, "
        f"y={y_min:.3f} to {y_max:.3f}"
    )

In [ ]:
for subdir in sorted(BASE_DIR.iterdir()):

    if not subdir.is_dir():
        continue

    print(f"Looking in directory {subdir}")

    files = sorted(f for f in subdir.rglob("*.nc") if f.is_file())

    print(f"Found {len(files)} files in {subdir}")

    if not files:
        print(f"No relevant files found, skipping {subdir}")
        continue

    print("Opening files...")

    ds = xr.open_mfdataset(
        files,
        combine="nested",
        concat_dim="time",
        parallel=True,
        chunks="auto",
    )

    full_data = ds[var].sortby("time")

    full_data = full_data.rename({"longitude": "central_lon", "latitude": "central_lat"})

    # ---------------------------------------------------------
    # Crop to example grid bounds
    # ---------------------------------------------------------
    if example_grid_path is not None:

        if "x" not in full_data.coords or "y" not in full_data.coords:
            raise ValueError(
                "Input data must contain 'x' and 'y' coordinates."
            )

        # Account for ascending/descending coordinates
        x_ascending = full_data["x"].values[0] < full_data["x"].values[-1]
        y_ascending = full_data["y"].values[0] < full_data["y"].values[-1]

        x_slice = (
            slice(x_min, x_max)
            if x_ascending
            else slice(x_max, x_min)
        )

        y_slice = (
            slice(y_min, y_max)
            if y_ascending
            else slice(y_max, y_min)
        )

        full_data = full_data.sel(
            x=x_slice,
            y=y_slice,
        )

        print(
            f"Cropped to "
            f"{full_data.sizes['y']} x {full_data.sizes['x']} grid"
        )

    # ---------------------------------------------------------
    # 6-hour means
    # ---------------------------------------------------------
    print("Resampling to 6-hourly data...")

    data_6h = full_data.resample(time="6h").mean()

    write_tasks = []

    for time in data_6h.time.values:

        timestamp = pd.Timestamp(time)

        year = timestamp.strftime("%Y")
        filename = timestamp.strftime("%Y%m%d_%H.nc")

        year_dir = OUT_DIR_6H / year
        year_dir.mkdir(parents=True, exist_ok=True)

        out_path = year_dir / filename

        if out_path.is_file():
            print(f"{out_path} already exists, skipping")
            continue

        data_out = (
            data_6h
            .sel(time=[time])
            .to_dataset(name=var_out)
        )

        task = data_out.to_netcdf(
            out_path,
            compute=False,
        )

        write_tasks.append(task)

    print(
        f"Computing and writing {len(write_tasks)} "
        f"6-hourly files in parallel..."
    )

    if write_tasks:
        with ProgressBar():
            dask.compute(
                *write_tasks,
                scheduler="threads",
                num_workers=8,
            )

    ds.close()

    print(f"Finished {subdir}")

In [ ]:
# Load all 6-hourly files
files = sorted(OUT_DIR_6H.glob("*/*.nc"))

if not files:
    raise FileNotFoundError(f"No NetCDF files found in {OUT_DIR_6H}")

print(f"Found {len(files)} 6-hourly files")

ds = xr.open_mfdataset(files, combine="by_coords").sortby("time")

# 24-hour accumulation periods starting at 06:00
daily = ds.resample(
    time="24h",
    origin="start_day",
    offset="6h",
    label="left",
    closed="left",
).sum()

# Save one file per day, labelled by start of accumulation period
for time in daily.time.values:
    timestamp = pd.Timestamp(time)

    year_dir = OUT_DIR_24H / timestamp.strftime("%Y")
    year_dir.mkdir(parents=True, exist_ok=True)

    out_path = year_dir / timestamp.strftime("%Y%m%d_%H.nc")

    if out_path.is_file():
        print(f"{out_path} already exists, skipping")
        continue

    print(f"Saving to {out_path}")

    daily.sel(time=[time]).to_netcdf(out_path)

ds.close()

print("Done")